# Different data formats

This notebook shows some examples for loading file formats from different battery testers as well as some "tweaking" possibilites provided by `cellpy`. We hope that, as time goes, a more complete set of instruments will be fully supported. Loading non-supported ("custom") file formats is explained in more detail [here](./07_custom_loaders.ipynb).

In [ ]:
import cellpy
from cellpy.utils import example_data, plotutils
from rich import print

In [ ]:
# Comment out this when running the notebook locally to get interactive plots (use notebook renderer locally if preferred)
import plotly.io as pio

pio.renderers.default = "json"

## Overview

To get an overview on all the implemented instruments/loaders:

In [ ]:
from cellpy.readers import data_structures as core

print(core.find_all_instruments().keys())

Some instruments have different types of `models` - for more details on those, have a look at the section on reading of *Maccor* data below.

Defining a simple utility-function to get a peek of the file in question:

In [ ]:
def head(f, n=5):
    print(f" {f.name} ".center(80, "-"))
    with open(f) as datafile:
        for j in range(n):
            line = datafile.readline()
            print(f"[{j + 1:02}] {line.strip()}")

## PEC CSV data

PEC testers do not seem to allow direct access to the raw data (database). However, data can be exported to csv-files from the graphical user interface. There might exist other solutions as well (let us know). 

`cellpy` contains a limited set of example data sets, among others, a csv-file exported from a run performed at a PEC tester. The example data can be downloaded to your PC using the `utils.example_data` module:

In [ ]:
p = example_data.pec_file_path()
print(f"{p.name=}")

Below we take a look at the first 35 lines of the example PEC csv-files.

If the file you want to load is not similar to this, either a custom loader must be made, or you can create an issue on GitHub (and maybe help in implementing the necessery modifications?). 

In [ ]:
head(p, 35)

### Loading the file

You can load the file using the `.get` method as usual. However, you will have to provide `cellpy` the name of the instrument (for this case it will be "pec_csv").

In [ ]:
c = cellpy.get(p, instrument="pec_csv", cycle_mode="full_cell")
plotutils.raw_plot(c, width=1200, height=400)

Once you have loaded the files, you can use all the common functionalities of `cellpy` (as described in other example notebooks), such as, e.g., looking at a summary plot:

In [ ]:
plotutils.summary_plot(c, y="capacities", width=1200, height=400)

## MACCOR

The implemented loader for exported data from Maccor is able to load several "file-morphologies" (so-called `models`). This illustrates one of the main weaknesses of not having direct access to the raw-data: the operator/user typically has the possibility select (consciously or not) how the final exported data file will look. This can be, for example, what to name the columns, what to use as delimiter, or what symbol to use as thousand seperator.

### Different models

You can get information about the different models for the loaders by looking at the instrument configurations. Here is an example of how to do it "programmatically":

1. Check which loaders are available for Maccor files:

In [ ]:
config = core.instrument_configurations("maccor")
print(config.keys())

2. Check which *models* are available for Maccor:

In [ ]:
print(config["maccor_txt"]["__all__"])

3. Have a closer look at a selected model configuration, here for model `THREE`:

In [ ]:
print(config["maccor_txt"]["THREE"])

Especially the `formatters` give valuable hints if a model is promising for your specific file or not:

In [ ]:
print(config["maccor_txt"]["THREE"]["config_params"].formatters)

Note that "config_params" is not a dictionary, but an instance of the ModelParameters class (so dot notation is needed).

### Loading the file
Now we are ready to look into loading an example Maccor file (included within cellpy's `utils.example_data` module):

In [ ]:
p = example_data.maccor_file_path()
print(f"{p.name=}")

In [ ]:
head(p, 10)

The file format for this file is handled by the model `THREE` in `cellpy`. Both, information on the instrument ("maccor_txt") and on the *model* ("THREE") has to be included when loading the data using the standard  `cellpy.get` method:

In [ ]:
c = cellpy.get(p, instrument="maccor_txt", model="THREE", cycle_mode="full_cell")

After loading the file, you are ready to use all common `cellpy` functionalities:

In [ ]:
print(f"Available cycles in the file: {c.get_cycle_numbers()}")
plotutils.raw_plot(c, width=1200, height=400)

In [ ]:
plotutils.summary_plot(c, y="capacities", width=1200, height=400, y_range=[0, 1000])

## NEWARE

Data from Neware testers will be improved soon. Currently, one `model` is implemented ("ONE"). Using the method described above for getting information, currently you will see three model names appear. The "default" is the one that will be picked if no model name is provided ("ONE" for now), while "UIO" is just a nick-name for the "ONE" model.

In [ ]:
config = core.instrument_configurations("neware")
print(config["neware_txt"]["__all__"])

Check the configuration for *model* `ONE`:

In [ ]:
print(config["neware_txt"]["ONE"])

In [ ]:
p = example_data.neware_file_path()
print(f"{p.name=}")

In [ ]:
c = cellpy.get(p, instrument="neware_txt", mass=2.09)

Notice that this loader (with the default model) uses the auto-formatting method. The method tries to find out type of delimiter and number of header rows automatically. You can override this by providing the values in the call yourself, for example `c.get(p, instrument="neware_txt", sep=",")`

In [ ]:
plotutils.raw_plot(c, width=1200, height=400)

In [ ]:
plotutils.summary_plot(c, y="capacities_gravimetric", width=1200, height=400, y_range=[0, 4000])

## Other

The `cellpy` team is working actively on implementing support for more instruments. If the file format is not too challenging, consider using a custom loader (see [custom loaders](07_custom_loaders.ipynb)).